In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../shipments_realistic.csv')
df.head()

,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,2012-07-07,2012-07-11,1.37,Viettel Post,Truck,7,5.0,99.0,...,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,2012-07-06,2012-07-10,2.60,J&T Express,Van,2,4.9,98.4,...,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,2012-07-04,2012-07-07,2.38,GHN,Motorbike,10,4.8,95.1,...,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,2012-07-05,2012-07-11,2.49,Viettel Post,Truck,8,5.0,96.3,...,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,2012-07-09,2012-07-16,25.79,BEST Express,Truck,10,4.6,95.7,...,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   shipper_id                566067 non-null  str    
 1   order_id                  566067 non-null  int64  
 2   ship_date                 566067 non-null  str    
 3   delivery_date             566067 non-null  str    
 4   shipping_fee              566067 non-null  float64
 5   shipper_company           566067 non-null  str    
 6   shipper_vehicle           566067 non-null  str    
 7   shipper_experience_years  566067 non-null  int64  
 8   shipper_rating            566067 non-null  float64
 9   delivery_success_rate     566067 non-null  float64
 10  average_delivery_time     566067 non-null  int64  
 11  working_shift             566067 non-null  str    
 12  join_date                 566067 non-null  str    
 13  shipper_name              566067 non-null  str    
 14 

In [4]:
# shipper_id là PK của hồ sơ shipper; order_id là FK trỏ về bảng orders (theo Dictionary)
# Kiểm tra trùng lặp theo cặp (shipper_id, order_id): mỗi shipper chỉ giao 1 đơn 1 dòng
print(df.duplicated(subset=['shipper_id', 'order_id']).sum())

0


In [5]:
# Chuyển 3 cột ngày sang kiểu datetime (errors='coerce': ngày sai sẽ thành NaT để dễ phát hiện)
df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')
# Kiểm tra thử có cái nào không đọc được không ?
print('Số ngày không đọc được:', df[['ship_date', 'delivery_date', 'join_date']].isna().sum())

Số ngày không đọc được: ship_date        0
delivery_date    0
join_date        0
dtype: int64


In [6]:
# Xem các đơn vị vận chuyển có giá trị nào bất thường không
df['shipper_company'].value_counts()

shipper_company
Ninja Van         99023
J&T Express       77814
Viettel Post      77808
BEST Express      77750
GHTK              70743
GHN               63854
Ahamove           56636
Shopee Express    42439
Name: count, dtype: int64

In [7]:
# Xem phương tiện giao hàng
df['shipper_vehicle'].value_counts()

shipper_vehicle
Motorbike    233672
Van          176837
Truck        155558
Name: count, dtype: int64

In [8]:
# Xem ca làm việc
df['working_shift'].value_counts()

working_shift
Morning      198346
Evening      190904
Afternoon    176817
Name: count, dtype: int64

In [9]:
# Xem giới tính
df['shipper_gender'].value_counts()

shipper_gender
Female    297357
Male      268710
Name: count, dtype: int64

In [10]:
# Xem trạng thái hôn nhân
df['shipper_marital_status'].value_counts()

shipper_marital_status
Married    290319
Single     275748
Name: count, dtype: int64

In [11]:
# Xem bằng cấp
df['shipper_education'].value_counts()

shipper_education
Bachelor       240442
High School    176940
College        148685
Name: count, dtype: int64

In [12]:
# Xem khu vực địa lý
df['region'].value_counts()

region
Central    233471
East       226632
West       105964
Name: count, dtype: int64

In [13]:
# Xem thành phố và quận huyện (chỉ đếm số lượng để không in dài)
print('Số thành phố :', df['city'].nunique())
print('Số quận huyện:', df['district'].nunique())
df['city'].value_counts().head()

Số thành phố : 34
Số quận huyện: 35


city
Son Tay      28518
Hoi An       28434
Nha Trang    28432
Viet Tri     28403
Tam Ky       28386
Name: count, dtype: int64

In [14]:
# Chuyển cột phân loại sang kiểu category để tiết kiệm bộ nhớ
for col in ['shipper_company', 'shipper_vehicle', 'working_shift', 'shipper_gender',
            'shipper_marital_status', 'shipper_education', 'city', 'region', 'district']:
    df[col] = df[col].astype('category')

In [15]:
# Dictionary: SĐT định dạng 9xxxxxxxxx (nguồn đang có đúng 9 chữ số, bắt đầu bằng 9)
# Đọc CSV thành số sẽ mất số 0 đầu -> giữ dạng chuỗi
df['shipper_phone'] = df['shipper_phone'].astype(str)
df[['shipper_id', 'shipper_name', 'shipper_phone']].head()

,shipper_id,shipper_name,shipper_phone
0,SHP00001,Bùi Văn Long,991476209
1,SHP00002,Trần Anh Khánh,959297982
2,SHP00003,Hoàng Thị Khánh,927142576
3,SHP00004,Trần Đức Vy,971617475
4,SHP00005,Trần Minh Cường,979196342


In [16]:
# Kiểm tra FK: order_id là khóa ngoại phải tồn tại trong bảng orders (theo Dictionary)
orders = pd.read_csv('../orders_enriched.csv')
print('Số order_id không tồn tại trong orders:', (~df['order_id'].isin(orders['order_id'])).sum())

Số order_id không tồn tại trong orders: 0


In [17]:
# Đối chiếu master data: bộ ba (city, region, district) phải có trong bảng geography
geo = pd.read_csv('../geography.csv')
check = df[['city', 'region', 'district']].drop_duplicates().merge(
    geo[['city', 'region', 'district']].drop_duplicates(), how='left', indicator=True)
print('Số bộ ba địa lý không có trong geography:', (check['_merge'] == 'left_only').sum())

Số bộ ba địa lý không có trong geography: 0


In [18]:
# Tổng kết trước khi xuất: không còn trùng, không còn null
print('Tổng số dòng     :', len(df))
print('Tổng số dòng trùng shipper_id, order_id  :', df.duplicated(subset=['shipper_id', 'order_id']).sum())
print('Tổng số null     :', df.isna().sum().sum())

Tổng số dòng     : 566067
Tổng số dòng trùng shipper_id, order_id  : 0
Tổng số null     : 0


In [19]:
df.to_csv('../SilverData/shipments_realistic.csv', index=False)